# 10-Level Anonymization System

Data anonymization sits at the heart of every modern privacy framework. GDPR (Article 5(1)(c)),
HIPAA's Safe Harbor method, and CCPA all demand that organizations minimize the personal data they
retain and protect what remains. The fundamental challenge is the **privacy-utility trade-off**:
stronger anonymization destroys more analytical value, while weaker techniques leave individuals
vulnerable to re-identification attacks.

Zipminator addresses this with a **graduated 10-level anonymization system**. Rather than forcing
a binary choice between raw data and total redaction, the engine lets data stewards choose the
precise level of protection appropriate for each column and each use case. Level 1 provides
lightweight one-way hashing suitable for audit logs; Level 10 applies Paillier homomorphic
encryption for computation on ciphertext. Each level maps to a pricing tier so that free-tier
users still get meaningful protection, while enterprise customers unlock the strongest guarantees.

| Tier | Levels | Intended Audience |
|------|--------|-------------------|
| Free | L1 -- L3 | Individual developers, open-source projects |
| Developer | L4 -- L5 | Startups, small teams with analytics needs |
| Pro | L6 -- L7 | Mid-size companies, regulated industries |
| Enterprise | L8 -- L10 | Banks, hospitals, government agencies |

This notebook walks through every level with executable code, explains the theory behind each
technique, and concludes with visualizations of the privacy-utility trade-off.

## Environment Setup

We begin by importing the Zipminator `AnonymizationEngine` and constructing a sample dataset
that mimics a typical HR or CRM table. The dataset contains six individuals with a mix of
direct identifiers (name, email, SSN, phone), quasi-identifiers (age, zipcode), and a
sensitive attribute (salary). This combination is deliberately chosen because research has
shown that as few as three quasi-identifiers (zip code, date of birth, sex) can uniquely
identify 87% of the U.S. population (Sweeney, 2000).

We also configure a dark-themed matplotlib style that matches the Zipminator quantum UI
palette. All plots in this notebook use OKLCH-derived hex colors for consistency with the
product design system.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from zipminator.crypto.anonymization import AnonymizationEngine

# -- Quantum dark theme for all plots ----------------------------------------
ZM_STYLE = {
    'figure.facecolor': '#0a0f1e', 'axes.facecolor': '#111827',
    'axes.edgecolor': '#334155', 'axes.labelcolor': '#e2e8f0',
    'axes.titleweight': 'bold', 'axes.grid': True,
    'grid.color': '#1e293b', 'grid.alpha': 0.5,
    'text.color': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'legend.facecolor': '#1e293b', 'legend.edgecolor': '#334155',
    'font.family': 'sans-serif', 'font.size': 11,
    'savefig.facecolor': '#0a0f1e', 'savefig.bbox': 'tight', 'savefig.dpi': 150,
}
plt.rcParams.update(ZM_STYLE)
ZM_CYAN = '#22d3ee'; ZM_VIOLET = '#a78bfa'; ZM_EMERALD = '#34d399'
ZM_AMBER = '#f59e0b'; ZM_ROSE = '#fb7185'; ZM_BLUE = '#3b82f6'

# -- Sample dataset ----------------------------------------------------------
engine = AnonymizationEngine()

df = pd.DataFrame({
    "name":    ["Alice Johnson", "Bob Smith", "Carol Williams",
                "Dave Brown", "Eve Davis", "Frank Miller"],
    "email":   ["alice@example.com", "bob@corp.net", "carol@hospital.org",
                "dave@bank.com", "eve@university.edu", "frank@gov.no"],
    "age":     [34, 52, 28, 45, 31, 67],
    "salary":  [85_000, 120_000, 72_000, 95_000, 68_000, 145_000],
    "zipcode": ["10001", "90210", "60601", "30301", "02101", "0150"],
    "phone":   ["212-555-0101", "310-555-0202", "312-555-0303",
                "404-555-0404", "617-555-0505", "+47-555-0606"],
    "ssn":     ["123-45-6789", "987-65-4321", "456-78-9012",
                "321-54-9876", "654-32-1098", "789-01-2345"],
})

print("Original Dataset (6 records, 7 columns):")
print(df.to_string(index=False))

---

## Level 1: SHA-256 Hashing

Cryptographic hashing is the simplest form of pseudonymization. Each value is passed through
the SHA-256 hash function, producing a fixed-length 64-character hexadecimal digest. The
transformation is **deterministic** (the same input always yields the same output) and
**one-way** (you cannot recover the original value from the hash alone).

**Use cases:**
- **Deduplication**: Two systems can compare hashed emails to find overlapping users without
  sharing the raw addresses.
- **Audit logs**: Store hashed identifiers so that log entries can be correlated without
  exposing PII to log aggregation platforms.
- **Data linkage**: Researchers can join datasets on hashed keys without seeing original values.

**Limitations:** SHA-256 is vulnerable to dictionary and rainbow-table attacks on low-entropy
inputs (e.g., names, phone numbers). For stronger guarantees, consider salted hashing or
moving to higher anonymization levels.

In [ ]:
# Level 1: SHA-256 Hashing (Free tier)
result_l1 = engine.apply_anonymization(df.copy(), columns=["name", "email", "ssn"], level=1)
print("L1 - SHA-256 Hashing (name, email, ssn):")
print(result_l1[["name", "email", "ssn"]].to_string(index=False))
print()
print(f"Hash length: {len(result_l1['name'].iloc[0])} characters (256 bits in hex)")
print(f"Deterministic check: same input -> same output = {result_l1['name'].iloc[0] == engine.apply_anonymization(df.copy(), columns=['name'], level=1)['name'].iloc[0]}")

---

## Level 2: Quantum-Random Replacement

At Level 2, every value is replaced with a 10-character string generated by the Zipminator
`QuantumRandom` engine. Unlike classical pseudo-random number generators (PRNGs) that rely on
deterministic algorithms seeded by system entropy, quantum random number generators (QRNGs)
derive randomness from inherently unpredictable quantum mechanical processes.

Why does the entropy source matter for anonymization? A PRNG with a compromised seed can be
replayed, allowing an attacker to reconstruct every replacement value and reverse the
anonymization. Quantum randomness is provably unpredictable under the laws of physics,
making replay attacks impossible regardless of computational power.

**Use cases:**
- Irreversible PII removal when no analytical value is needed from the column.
- Replacing direct identifiers before sharing data with third-party vendors.
- Generating placeholder values for staging and QA environments.

**Key property:** Non-deterministic. Running Level 2 twice on the same input produces
different outputs every time.

In [ ]:
# Level 2: Quantum-Random Replacement (Free tier)
result_l2 = engine.apply_anonymization(df.copy(), columns=["name", "email"], level=2)
print("L2 - Quantum-Random Replacement:")
print(result_l2[["name", "email", "age", "salary"]].to_string(index=False))
print()

# Demonstrate non-determinism: run again, get different values
result_l2b = engine.apply_anonymization(df.copy(), columns=["name"], level=2)
print(f"Non-deterministic: first run = {result_l2['name'].iloc[0]}, second run = {result_l2b['name'].iloc[0]}")
print(f"Values differ: {result_l2['name'].iloc[0] != result_l2b['name'].iloc[0]}")

---

## Level 3: Deterministic Tokenization

Tokenization replaces sensitive values with non-sensitive **tokens** that serve as surrogate
identifiers. Unlike encryption, tokenization does not use a mathematical relationship between
the original value and the token; instead, a lookup table maps each unique value to a unique
token. The token itself reveals nothing about the original data.

Zipminator's Level 3 uses deterministic tokenization based on MD5 digests (truncated to 8 hex
characters) prefixed with `TOKEN_`. This means the same input always maps to the same token,
enabling cross-system joins and record linkage without exposing PII.

**Tokenization vs. encryption:**
- Encryption preserves a mathematical relationship; anyone with the key can decrypt.
- Tokenization stores the mapping in a vault; the token itself is meaningless.
- Tokenization is often preferred in PCI-DSS environments for credit card data.

**Use cases:**
- Payment processing: replace card numbers with tokens for downstream analytics.
- Cross-system joins: two departments can match records on tokens without sharing names.
- Data warehouse pseudonymization with reversibility (if the token map is retained).

In [ ]:
# Level 3: Deterministic Tokenization (Free tier)
result_l3 = engine.apply_anonymization(df.copy(), columns=["name", "email", "ssn"], level=3)
print("L3 - Deterministic Tokenization:")
print(result_l3[["name", "email", "ssn"]].to_string(index=False))
print()

# Show that the token map is stored and reversible
print("Token map for 'name' column (stored internally):")
for original, token in engine._token_maps.get("name", {}).items():
    print(f"  {original:20s} -> {token}")

---

## Level 4: Generalization

Generalization reduces the precision of data values so that individual records become
indistinguishable from others in the same group. For numeric columns, values are bucketed
into fixed-width ranges (e.g., salary 85,000 becomes "80000-90000"). For text columns,
values are replaced with a category label derived from the first character.

This technique is foundational to many privacy frameworks because it preserves **statistical
utility** while removing **individual specificity**. A generalized dataset still supports
aggregate queries ("What is the average salary in the 80K-90K bracket?") but prevents
pinpointing any single person's exact compensation.

Generalization is a core building block of k-anonymity (Level 8) and is the primary
mechanism recommended by HIPAA's Safe Harbor de-identification method for geographic
and date variables.

**Use cases:**
- Demographic analysis where exact values are unnecessary.
- Publishing research datasets that must pass IRB review.
- Reducing granularity of location data (full address to city or region).

In [ ]:
# Level 4: Generalization (Developer tier)
# Text columns -> category labels
result_l4_text = engine.apply_anonymization(df.copy(), columns=["name", "email"], level=4)
print("L4 - Generalization (text columns):")
print(result_l4_text[["name", "email"]].to_string(index=False))
print()

# Numeric columns -> range buckets
result_l4_num = engine.apply_anonymization(df.copy(), columns=["age", "salary"], level=4)
print("L4 - Generalization (numeric columns):")
print(result_l4_num[["name", "age", "salary"]].to_string(index=False))
print()
print("Notice: Alice's age 34 -> '30-40', salary 85000 -> '80000-90000'")
print("The data is still analytically useful but no longer pinpoints individuals.")

---

## Level 5: Suppression

Suppression is the most aggressive form of data minimization: the entire column is replaced
with `NA` (null) values. No information from the original column survives. This directly
implements the **data minimization principle** enshrined in GDPR Article 5(1)(c), which
states that personal data must be "adequate, relevant, and limited to what is necessary."

When should you suppress rather than anonymize? Suppression is appropriate when:
- The column is collected for a purpose that no longer applies (purpose limitation).
- The column contains highly sensitive data (e.g., SSN) that no downstream analysis requires.
- Regulatory guidance explicitly calls for deletion (e.g., CCPA right-to-delete requests).

Suppression is also used as a fallback in k-anonymity algorithms when generalization alone
cannot achieve the required k-value without distorting the data beyond usefulness.

**Trade-off:** Maximum privacy, zero utility. The column is effectively dropped.

In [ ]:
# Level 5: Suppression (Developer tier)
result_l5 = engine.apply_anonymization(df.copy(), columns=["ssn", "phone"], level=5)
print("L5 - Suppression (ssn, phone removed):")
print(result_l5.to_string(index=False))
print()
print(f"Null count in 'ssn' column: {result_l5['ssn'].isna().sum()} / {len(result_l5)}")
print("The SSN and phone columns are fully suppressed -- no PII remains.")

---

## Level 6: Quantum Noise Injection

Noise injection adds controlled randomness to data values, preserving the general statistical
shape of a distribution while preventing exact value recovery. This is a practical
implementation of ideas from **differential privacy**, where mathematically calibrated noise
provides formal privacy guarantees.

Zipminator's Level 6 applies noise differently depending on column type:
- **Numeric columns**: Each value is perturbed by up to +/-10% of its magnitude, with the
  perturbation direction and magnitude determined by quantum randomness.
- **Text columns**: Values are replaced entirely with 10-character quantum-random strings
  (since adding "noise" to text has no meaningful statistical interpretation).

The quantum entropy source ensures that the noise pattern cannot be predicted or reversed,
even by an adversary with knowledge of the algorithm and parameters. Classical noise
injection using PRNGs is vulnerable to seed-recovery attacks; quantum noise is not.

**Use cases:**
- Publishing salary statistics where aggregate trends must be preserved.
- Sharing sensor data with third parties while protecting exact readings.
- Training machine learning models on perturbed data (noise-aware training).

In [ ]:
# Level 6: Quantum Noise Injection (Pro tier)
result_l6_num = engine.apply_anonymization(df.copy(), columns=["salary"], level=6)
print("L6 - Quantum Noise Injection (salary):")
comparison = pd.DataFrame({
    "name": df["name"],
    "original_salary": df["salary"],
    "noisy_salary": result_l6_num["salary"].round(0).astype(int),
})
comparison["delta"] = comparison["noisy_salary"] - comparison["original_salary"]
comparison["pct_change"] = ((comparison["delta"] / comparison["original_salary"]) * 100).round(2)
print(comparison.to_string(index=False))
print()
print(f"Mean original: ${df['salary'].mean():,.0f}")
print(f"Mean noisy:    ${result_l6_num['salary'].mean():,.0f}")
print("Aggregate statistics are approximately preserved; individual values shift.")

---

## Level 7: Synthetic Data Generation

Synthetic data replaces every value with a **realistic but entirely fabricated** alternative.
Zipminator uses the Faker library to generate contextually appropriate replacements: names
become other plausible names, emails become valid-looking email addresses, and so on.

Synthetic data occupies a unique position in the privacy landscape. Because no synthetic
record corresponds to any real individual, the dataset falls outside the scope of most
privacy regulations. GDPR Recital 26 explicitly excludes information that "does not relate
to an identified or identifiable natural person" from its definition of personal data.

The challenge is ensuring that synthetic data preserves the **statistical properties** of
the original: distributions, correlations, and edge cases should be faithfully reproduced.
Zipminator's Level 7 focuses on column-level synthesis (each column is replaced
independently). For correlated multi-column synthesis, consider pairing this with external
tools like SDV (Synthetic Data Vault).

**Use cases:**
- QA and staging environments that need realistic data without PII exposure.
- Sharing demo datasets with prospective customers.
- Training ML models when real data cannot leave the secure enclave.

In [ ]:
# Level 7: Synthetic Data (Pro tier)
result_l7 = engine.apply_anonymization(df.copy(), columns=["name", "email", "phone", "ssn"], level=7)
print("L7 - Synthetic Data (Faker-generated replacements):")
print(result_l7[["name", "email", "phone", "ssn"]].to_string(index=False))
print()
print("Every value is fabricated. No record maps to a real person.")
print("Non-PII columns (age, salary, zipcode) remain untouched:")
print(result_l7[["name", "age", "salary", "zipcode"]].to_string(index=False))

---

## Level 8: k-Anonymity

k-Anonymity is a formal privacy model introduced by Latanya Sweeney (2002). A dataset
satisfies **k-anonymity** if every combination of quasi-identifier values appears in at
least *k* records. In other words, any individual is indistinguishable from at least
*k* - 1 others based on the quasi-identifier columns alone.

**Formal definition:** A table *T* satisfies k-anonymity with respect to quasi-identifiers
*Q* = {q_1, q_2, ..., q_n} if for every record *r* in *T*, there exist at least *k* - 1
other records in *T* that are identical to *r* on all attributes in *Q*.

**Quasi-identifiers** are attributes that are not direct identifiers (like SSN) but can be
combined to re-identify individuals. Classic examples: zip code + date of birth + gender.
Zipminator achieves k-anonymity by generalizing quasi-identifiers (numeric to ranges,
text to category prefixes) until the k-threshold is met.

**Limitations:** k-Anonymity does not protect against attribute disclosure if the sensitive
attribute lacks diversity within an equivalence class. Extensions like l-diversity and
t-closeness address this, and are available at Level 9+.

**Use cases:**
- Publishing medical research datasets (HIPAA Expert Determination often requires k >= 5).
- Government census microdata releases.
- Any dataset shared with external researchers.

In [ ]:
# Level 8: k-Anonymity (Enterprise tier, k=5)
result_l8 = engine.apply_anonymization(df.copy(), columns=["name", "zipcode"], level=8)
print("L8 - k-Anonymity (quasi-identifiers generalized, k=5):")
print(result_l8.to_string(index=False))
print()
print("Quasi-identifiers are generalized so that each combination")
print("appears in at least k=5 records (with sufficient data).")
print("In this 6-row sample, generalization is aggressive to meet the threshold.")

---

## Level 9: Differential Privacy

Differential privacy (DP) provides the strongest formal privacy guarantee in the
anonymization literature. Introduced by Dwork et al. (2006), it ensures that the output
of a computation is statistically indistinguishable whether or not any single individual's
data is included in the input.

**Formal definition:** A randomized mechanism *M* satisfies (epsilon, delta)-differential
privacy if for all datasets *D_1* and *D_2* differing in at most one record, and for all
sets of outputs *S*:

    Pr[M(D_1) in S] <= exp(epsilon) * Pr[M(D_2) in S] + delta

The parameter **epsilon** controls the privacy budget: smaller epsilon means stronger privacy
but more noise. Typical values range from 0.1 (very private) to 10 (low privacy). The
parameter **delta** allows for a small probability of catastrophic failure and is usually
set to a value much smaller than 1/n.

Zipminator's Level 9 applies the **Laplace mechanism** to numeric columns. The noise scale
is calibrated to the column's sensitivity (max - min) divided by epsilon. For text columns,
a randomized response mechanism replaces values probabilistically.

**Use cases:**
- Federal statistics agencies (the U.S. Census Bureau uses DP for the 2020 Census).
- Tech companies publishing aggregate usage metrics (Apple, Google use local DP).
- Any scenario requiring a mathematical privacy proof.

In [ ]:
# Level 9: Differential Privacy (Enterprise tier, epsilon=1.0)
result_l9 = engine.apply_anonymization(df.copy(), columns=["salary", "age"], level=9)
print("L9 - Differential Privacy (Laplace noise on salary, age):")
dp_comparison = pd.DataFrame({
    "name": df["name"],
    "orig_salary": df["salary"],
    "dp_salary": result_l9["salary"].round(0).astype(int),
    "orig_age": df["age"],
    "dp_age": result_l9["age"].round(1),
})
print(dp_comparison.to_string(index=False))
print()
print(f"Epsilon = 1.0 (moderate privacy).")
print(f"Salary sensitivity = {df['salary'].max() - df['salary'].min():,} (range of column)")
print(f"Noise scale (salary) = sensitivity / epsilon = {(df['salary'].max() - df['salary'].min()) / 1.0:,.0f}")
print("Smaller epsilon -> larger noise -> stronger privacy guarantee.")

---

## Level 10: Paillier Homomorphic Encryption / OTP

Level 10 provides the strongest protection in the Zipminator system. For numeric data, it
applies **Paillier homomorphic encryption**, which allows arithmetic operations (addition
and scalar multiplication) to be performed directly on ciphertext without decryption.
This means a cloud server can compute aggregate statistics on encrypted salaries without
ever seeing the plaintext values.

**Homomorphic encryption basics:** Given ciphertexts `Enc(a)` and `Enc(b)`, the Paillier
cryptosystem supports:
- `Enc(a) * Enc(b) = Enc(a + b)` (additive homomorphism)
- `Enc(a)^k = Enc(a * k)` (scalar multiplication)

For non-numeric (text) data, Level 10 applies a one-time pad (OTP) style transformation
using SHA-256, producing an `ENC_` prefixed ciphertext. If the `phe` library is not
installed, Level 10 gracefully degrades to Level 9 (differential privacy).

**Use cases:**
- Secure multi-party computation: multiple hospitals compute aggregate disease statistics
  without sharing patient records.
- Encrypted search: query encrypted databases without decrypting the index.
- Maximum-security data vaults for classified or defense-related information.

In [ ]:
# Level 10: Paillier/OTP (Enterprise tier)
result_l10 = engine.apply_anonymization(df.copy(), columns=["name", "email"], level=10)
print("L10 - Paillier / OTP (maximum protection):")
print(result_l10[["name", "email"]].to_string(index=False))
print()

# Demonstrate on numeric column if phe is available
try:
    from phe import paillier as _phe_check
    result_l10_num = engine.apply_anonymization(df.copy(), columns=["salary"], level=10)
    print("L10 - Paillier on salary (numeric, homomorphic):")
    print(f"  Type of encrypted value: {type(result_l10_num['salary'].iloc[0]).__name__}")
    print(f"  Original salary[0]: {df['salary'].iloc[0]:,}")
    # Decrypt to verify correctness
    decrypted = engine.private_key.decrypt(result_l10_num['salary'].iloc[0])
    print(f"  Decrypted salary[0]: {decrypted:,.0f}")
    print(f"  Round-trip verified: {abs(decrypted - df['salary'].iloc[0]) < 0.01}")
except ImportError:
    print("  [phe library not installed -- Level 10 degrades to Level 9 for numeric data]")
    print("  Install with: uv pip install phe")

---

## Privacy-Utility Trade-off Visualization

The following chart plots each anonymization level on two axes: **privacy score** (how
strongly it protects individual records) and **data utility score** (how much analytical
value remains after transformation). The scores are normalized to a 0-10 scale.

The ideal technique would sit in the upper-right corner (maximum privacy, maximum utility),
but information theory tells us this is impossible for non-trivial datasets. Every
anonymization technique trades utility for privacy. The art of privacy engineering is
choosing the right point on this curve for each use case.

In [ ]:
# Privacy-Utility Trade-off Visualization
levels = list(range(1, 11))
labels = [
    "L1: SHA-256", "L2: Q-Random", "L3: Tokenize", "L4: Generalize",
    "L5: Suppress", "L6: Q-Noise", "L7: Synthetic", "L8: k-Anon",
    "L9: Diff. Privacy", "L10: Paillier"
]
privacy_scores  = [2.5, 5.0, 4.0, 4.5, 8.0, 6.5, 7.0, 8.5, 9.0, 10.0]
utility_scores  = [3.0, 1.0, 7.5, 6.5, 0.5, 5.5, 7.0, 5.0, 3.5, 2.0]

tier_colors = {
    "Free": ZM_CYAN, "Developer": ZM_EMERALD,
    "Pro": ZM_AMBER, "Enterprise": ZM_ROSE,
}
tier_map = ["Free", "Free", "Free", "Developer", "Developer",
            "Pro", "Pro", "Enterprise", "Enterprise", "Enterprise"]
colors = [tier_colors[t] for t in tier_map]

fig, ax = plt.subplots(figsize=(12, 7))

# Scatter points
for i in range(10):
    ax.scatter(utility_scores[i], privacy_scores[i], c=colors[i],
               s=220, zorder=5, edgecolors='white', linewidths=0.8)

# Annotations
offsets = [
    (12, -18), (-15, 14), (12, 10), (12, -18), (-20, 14),
    (12, 12), (12, -18), (-20, -18), (12, 12), (-20, 14),
]
for i in range(10):
    ax.annotate(
        labels[i], (utility_scores[i], privacy_scores[i]),
        textcoords="offset points", xytext=offsets[i],
        fontsize=9, color=colors[i], fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=colors[i], lw=0.8),
    )

# Pareto frontier (approximate)
pareto_x = [0.5, 1.0, 3.5, 5.5, 6.5, 7.0, 7.5]
pareto_y = [8.0, 9.0, 9.0, 6.5, 4.5, 7.0, 4.0]
pareto_sorted = sorted(zip(pareto_x, pareto_y))
px, py = zip(*pareto_sorted)
ax.plot(px, py, '--', color=ZM_VIOLET, alpha=0.4, linewidth=1.5, label='Pareto frontier (approx.)')

# Axis labels and formatting
ax.set_xlabel('Data Utility Score', fontsize=13, fontweight='bold')
ax.set_ylabel('Privacy Protection Score', fontsize=13, fontweight='bold')
ax.set_title('Zipminator Anonymization: Privacy vs. Utility Trade-off', fontsize=15, pad=15)
ax.set_xlim(-0.5, 9.5)
ax.set_ylim(0, 11)

# Tier legend
legend_patches = [mpatches.Patch(color=c, label=t) for t, c in tier_colors.items()]
ax.legend(handles=legend_patches, loc='lower left', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

---

## Tier Mapping Heatmap

The heatmap below shows which anonymization levels are available at each pricing tier.
Each tier includes all levels from lower tiers, so Enterprise customers have access to
all 10 levels. This design ensures that upgrading never reduces capability.

In [ ]:
# Tier Mapping Heatmap
tiers = ["Free", "Developer", "Pro", "Enterprise"]
level_labels = [f"L{i}" for i in range(1, 11)]

# Access matrix: 1 = available, 0 = locked
access = np.array([
    [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],  # Free
    [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],  # Developer
    [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],  # Pro
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # Enterprise
])

fig, ax = plt.subplots(figsize=(12, 4))

# Custom colormap: dark for locked, cyan for available
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#1e293b', '#0e7490'])

im = ax.imshow(access, cmap=cmap, aspect='auto', vmin=0, vmax=1)

# Labels
ax.set_xticks(range(10))
ax.set_xticklabels(level_labels, fontsize=11, fontweight='bold')
ax.set_yticks(range(4))
ax.set_yticklabels(tiers, fontsize=12, fontweight='bold')
ax.set_xlabel('Anonymization Level', fontsize=13, fontweight='bold')
ax.set_title('Level Availability by Pricing Tier', fontsize=15, pad=12)

# Cell annotations
for i in range(4):
    for j in range(10):
        symbol = '\u2713' if access[i, j] else '\u2014'
        color = '#e2e8f0' if access[i, j] else '#475569'
        ax.text(j, i, symbol, ha='center', va='center',
                fontsize=14, color=color, fontweight='bold')

# Grid lines
ax.set_xticks(np.arange(-0.5, 10, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(which='minor', color='#334155', linewidth=0.5)
ax.tick_params(which='minor', size=0)

plt.tight_layout()
plt.show()

---

## Trade-off Summary

| Level | Technique | Privacy | Utility | Reversible | Tier | Best For |
|:-----:|-----------|:-------:|:-------:|:----------:|:----:|----------|
| L1 | SHA-256 Hashing | Low | Low | No | Free | Deduplication, audit logs |
| L2 | Quantum-Random Replacement | Medium | None | No | Free | Complete PII removal |
| L3 | Deterministic Tokenization | Medium | High | With map | Free | Cross-system joins, PCI-DSS |
| L4 | Generalization | Medium | Medium | No | Developer | Demographic analysis, IRB datasets |
| L5 | Suppression | High | None | No | Developer | GDPR data minimization, right-to-delete |
| L6 | Quantum Noise Injection | High | Medium | No | Pro | Statistical analysis, ML training |
| L7 | Synthetic Data | High | High | No | Pro | QA environments, demos, ML training |
| L8 | k-Anonymity (k=5) | Very High | Medium | No | Enterprise | Research datasets, HIPAA Safe Harbor |
| L9 | Differential Privacy | Very High | Low | No | Enterprise | Census data, regulatory compliance |
| L10 | Paillier / OTP | Maximum | Compute-only | With key | Enterprise | Secure MPC, encrypted analytics |

---

## Choosing the Right Level: A Decision Framework

Selecting an anonymization level depends on three factors: the **sensitivity** of the data,
the **analytical requirements** of the downstream consumer, and the **regulatory context**.

**By regulation:**
- **GDPR (EU):** Pseudonymization (L1-L3) is recognized but still counts as personal data.
  True anonymization (L7+ synthetic data, L9 differential privacy) may fall outside GDPR
  scope if re-identification is reasonably impossible.
- **HIPAA (US):** Safe Harbor requires suppression or generalization of 18 identifier types.
  L4-L5 for quasi-identifiers, L8 (k >= 5) for research datasets.
- **CCPA (California):** Right-to-delete maps directly to L5 suppression. De-identification
  at L7+ satisfies the "reasonably" standard.
- **DORA (EU/Norway):** Article 6 requires documented encryption policies. L6+ with quantum
  entropy provides auditable, forward-secure protection.

**By use case:**
- Internal analytics: L4 (generalization) preserves aggregate trends.
- Third-party data sharing: L7 (synthetic) or L9 (differential privacy).
- Secure computation across organizations: L10 (Paillier homomorphic encryption).
- Audit and compliance logging: L1 (SHA-256 hashing).

When in doubt, start at a higher level and relax only if the downstream consumer
demonstrates a legitimate need for more granular data.